The below code is having the raw code for the word count application.

In this notebook we have a simple batch data read and write later enhanced with the class and method approach


In [0]:
file_directry ='/Volumes/workspace/word_count/sample//text/text_data_1.txt'

In [0]:
lines =spark.read.text(file_directry)
lines.display()

In [0]:
from pyspark.sql.functions import split,explode,trim, lower

raw_words=lines.select(explode(split(trim(lines.value)," ")).alias('words'))
raw_words.display()
good_words=raw_words.select(lower(trim(raw_words.words)).alias('good_words'))
good_words.display()

split just makes the list of the output and when we use the explode it puts the values into new rows.

In [0]:
word_counts=good_words.groupBy("good_words").count()
word_counts.display()

In [0]:
word_counts.write.format('delta').saveAsTable('word_counts_test1')

In [0]:
%sql
select * from word_counts_test1

in the below code, class and methods are implimented for following the standards.



In [0]:
class word_counts():
    def __init__(self):
        self.base_directory='/Volumes/workspace/word_count/sample//text/text_data_1.txt'
        
    def read_file(self):
        lines =spark.read.text(self.base_directory)
        return lines
    def quality_words(self,lines):
        from pyspark.sql.functions import split,explode,split,lower,trim
        raw_words=lines.select(explode(split(lines.value," ")).alias('words'))
        good_words=raw_words.select(lower(trim(raw_words.words)).alias('good_words'))
        return good_words
    def word_counts(self,good_words):
        word_counts=good_words.groupBy("good_words").count()
        return word_counts
    def final_write(self,word_counts):
        word_counts.write.format('delta').mode('overwrite').saveAsTable('word_counts_class')
    def run_load(self):
        print("Executing the code ",end=" ")
        lines=self.read_file()
        good_words=self.quality_words(lines)
        word_counts=self.word_counts(good_words)
        self.final_write(word_counts)
        print("Done")
    def assertcount(self,expected_count):
        self.run_load()
        print('Starting the test --- ', end=" ")
        actual_count=spark.sql("select count from word_counts_class where good_words='on'").collect()[0][0]
        assert actual_count==expected_count ,f"Test failed!! the count is {actual_count} but the provided count is {expected_count}"
        print('Test completed')

        
        

the alias should be written inside the select statement
after all the methods, we can use the extra method just to call the other methods and complete the execution like below

Assert function is when the given condition is failed then it will fail the job with the msg, while if it passes the execution will be complted.


In [0]:
word_counts_obj=word_counts()
# word_counts_obj.run_load()
word_counts_obj.assertcount(expected_count=3)

I can also call each methods seperately and execute as required
Only the __init__ will be executed by default

The same class can be called in different notebooks but have to use the %run before using the methods



In [0]:
word_counts_test=word_counts()
lines =word_counts_test.read_file()
good_words=word_counts_test.quality_words(lines)
word_count=word_counts_test.word_counts(good_words)
word_count.display()
word_counts_test.final_write(word_count)